In [1]:
import os
import sys
import uuid
import threading
from pathlib import Path
from typing import Optional
from dotenv import load_dotenv
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse
from pydantic import BaseModel
import OCR.utils.ovo_svm as ovo_svm

sys.modules["ovo_svm"] = ovo_svm

from Transformer import Transformer
from Storage.CustomVectorStore import CustomVectorStore
from Storage.HNSW import HNSWVectorStore
from Storage.SQL.Repositories.VideoRepository import VideoRepository
from Storage.SQL.Repositories.VRDRepository import VRDRepository
from Storage.SQL.Repositories.ObjectRepository import ObjectRepository
from Storage.SQL.Repositories.OCRRepository import OCRRepository
from Storage.SQL.DatabaseClient import SessionLocal
from Storage.SQL.Models.Object import Object as ObjectModel
from Storage.SQL.Models.ObjectVideo import ObjectVideo
from Storage.SQL.Models.VRDSubject import VRDSubject
from Storage.SQL.Models.VRDPredicate import VRDPredicate
from Storage.SQL.Models.VRDObject import VRDObject
from Storage.SQL.Models.VRDVideo import VRDVideo
from Storage.SQL.Models.Video import Video as VideoModel
import gc
import torch
from visual.SceneSegmenter import SceneSegmenter
from OCR.src.OCR import OCR
from visual.faster_rcnn.ObjectDetector import ObjectDetector
from visual.VRD import VRD
from audio.ASR import ASR
from audio.SentenceSegmenter import SentenceSegmentation

f:\CUFE\4th year\GP\vidseek\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
f:\CUFE\4th year\GP\vidseek\visual\VRD.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
def _run_pipeline(video_path: str,
                   detector: str = 'craft', recognizer: str = 'easyocr'):

    video_id = 404
    json_folder = "./json_outputs"
    print("Scene segmentation...")
    segmenter = SceneSegmenter(video_path)
    segmenter.segment_video()
    frames = segmenter.get_frames_with_metadata()
    print(f"Extracted {len(frames)} frames from video {video_path}")
    del segmenter; gc.collect()
    print("OCR processing...")
    ocr = OCR(frames, video_path, video_id, detector=detector, recognizer=recognizer)
    ocr.process_frames()
    ocr_index = ocr.get_inverted_index()
    print(f"OCR inverted index for video {video_id}: {ocr_index}")
    del ocr; gc.collect()
    # update("Object detection...")
    # obj_det = ObjectDetector(video_path, frames)
    # obj_det.detect_objects()
    # ObjectRepository().save_from_inverted_index(obj_det.get_inverted_index(), video_id)
    # del obj_det; gc.collect()
    # update("Visual relationship detection...")
    # vrd = VRD(frames=frames, video_path=video_path, api_key=os.getenv("GEMINI_TOKEN"))
    # vrd.detect_relationships()
    # VRDRepository().save_from_inverted_index(vrd.get_inverted_index(), video_id)
    # del vrd
    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()
    # gc.collect()
    # update("Audio transcription...")
    # asr = ASR(video_path=video_path, model_name="openai/whisper-small")
    # asr.transcribe(task="translate")
    # transcription_path = os.path.join(json_folder, f"{video_id}_transcription.json")
    # asr.save_transcription(transcription_path)
    # del asr
    # if torch.cuda.is_available():
    #     torch.cuda.empty_cache()
    # gc.collect()
    # update("Sentence segmentation...")
    # seg = SentenceSegmentation(
    #     video_path=video_path,
    #     transcript_json=transcription_path,
    #     similarity_threshold=0.75,
    # )
    # seg.segment()
    # transcript_segments = seg.segments
    # del seg; gc.collect()
    # update("Embedding and storing...")
    # transformer = Transformer(ocr_index, transcript_segments)
    # transformer.transform()
    # hnsw_store = HNSWVectorStore()
    # transformer.save_embeddings(hnsw_store)
    # hnsw_store.commit()  # flush vectors + graph to disk atomically
    print("Done")

In [4]:
path = "./videos/3.mp4"
_run_pipeline(path)

Scene segmentation...
Extracted 0 frames from video ./videos/3.mp4
OCR processing...


KeyboardInterrupt: 